In [3]:
!pip install -q langchain langchain-core langchain-classic langchain-community langchain-huggingface langchain-text-splitters

In [4]:
!pip install -q sentence-transformers faiss-cpu flashrank transformers torch rank_bm25 numpy

print("=" * 60)

# Indexing

In [ ]:
- document extraction 
- Chunking(logical block)
- Embeddings
- vector store
------------------------------------------------


In [1]:
!pip install -q langchain langchain-core langchain-classic langchain-community langchain-huggingface langchain-text-splitters
!pip install -q sentence-transformers faiss-cpu flashrank transformers torch rank_bm25 numpy

print("=" * 60)
print("ALL PACKAGES INSTALLED SUCCESSFULLY!")
print("=" * 60)

ALL PACKAGES INSTALLED SUCCESSFULLY!


In [3]:
import os
import warnings
import numpy as np
from pprint import pprint

# ── Compatibility patch for langchain v1.x ──
# langchain_core still references langchain.debug / langchain.verbose
# internally, but langchain v1.x removed these module-level attributes.
# Setting them here prevents AttributeError at runtime.
# See: https://github.com/langchain-ai/langchain/issues/19278
import langchain
for attr in ("debug", "verbose", "llm_cache"):
    if not hasattr(langchain, attr):
        setattr(langchain, attr, False)
warnings.filterwarnings("ignore")

# LangChain Core
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda

# Text Splitting
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings & Vector Store
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# LLM (transformers v5.x: "text2text-generation" was removed, use "text-generation")
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline

print("All imports successful!")
print(f"NumPy version: {np.__version__}")
print(f"LangChain version: {langchain.__version__}")
import transformers
print(f"Transformers version: {transformers.__version__}")
import sentence_transformers
print(f"Sentence-Transformers version: {sentence_transformers.__version__}")

All imports successful!
NumPy version: 2.3.5
LangChain version: 1.2.15
Transformers version: 5.5.4
Sentence-Transformers version: 5.4.1


In [4]:
sample_text = """
Artificial Intelligence: A Comprehensive Overview

Section 1: What is Artificial Intelligence?
Artificial Intelligence (AI) is the simulation of human intelligence processes by computer systems. These processes include learning (acquiring information and rules for using it), reasoning (using rules to reach approximate or definite conclusions), and self-correction. AI was founded as an academic discipline in 1956 at the Dartmouth Conference, and has experienced several waves of optimism, followed by disappointment and loss of funding known as "AI winters". The field has seen remarkable progress since 2012 due to advances in computing power and availability of large datasets.

Section 2: Machine Learning Fundamentals
Machine Learning (ML) is a subset of AI that enables systems to learn and improve from experience without being explicitly programmed. There are three main types of machine learning. First, Supervised Learning where the model learns from labeled training data to make predictions. Examples include classification (spam detection, image recognition) and regression (price prediction, weather forecasting). Second, Unsupervised Learning where the model finds hidden patterns in unlabeled data. Common techniques include clustering (customer segmentation) and dimensionality reduction (PCA). Third, Reinforcement Learning where an agent learns by interacting with an environment, receiving rewards for good actions and penalties for bad ones. Applications include game playing (AlphaGo) and robotics.

Section 3: Deep Learning and Neural Networks
Deep Learning is a subset of machine learning that uses artificial neural networks with multiple layers to model complex patterns. A neural network consists of layers of interconnected nodes (neurons) that process information. Key architectures include Convolutional Neural Networks (CNNs) which are designed for processing grid-like data such as images, Recurrent Neural Networks (RNNs) which handle sequential data like text and time series, and Transformers which use self-attention mechanisms and have revolutionized NLP since 2017. The Transformer architecture, introduced in the paper "Attention is All You Need", eliminated the need for recurrence and convolution, enabling much more efficient parallel processing.

Section 4: Natural Language Processing
Natural Language Processing (NLP) is a branch of AI focused on the interaction between computers and human language. Key NLP tasks include sentiment analysis (determining if text is positive or negative), named entity recognition (identifying people, places, organizations), machine translation (converting text between languages), text summarization (creating concise summaries of long documents), and question answering (finding answers to questions from a given context). Modern NLP heavily relies on large language models like BERT, GPT, T5, and LLaMA which are pre-trained on massive text corpora using self-supervised learning.

Section 5: Computer Vision
Computer Vision enables machines to interpret and make decisions based on visual data. Major applications include facial recognition for security and authentication, autonomous vehicles for detecting roads, obstacles, and traffic signs, medical image analysis for detecting tumors and diseases from X-rays and MRIs, and quality inspection in manufacturing for identifying defects. Deep learning, particularly CNNs like ResNet and EfficientNet, has dramatically improved computer vision accuracy since AlexNet won the ImageNet competition in 2012.

Section 6: Retrieval-Augmented Generation (RAG)
Retrieval-Augmented Generation (RAG) is a technique that enhances large language models by retrieving relevant information from external knowledge bases before generating responses. RAG addresses key limitations of LLMs such as knowledge cutoff dates, hallucination, and lack of domain-specific knowledge. The RAG process involves three main steps: first, indexing where documents are split into chunks, converted to embeddings, and stored in a vector database; second, retrieval where a user query is embedded and similar document chunks are found using similarity search; and third, generation where the retrieved context is combined with the query and fed to an LLM to produce an accurate answer.

Section 7: Vector Databases and Embeddings
Vector databases store data as high-dimensional vectors, enabling efficient similarity search. Text embeddings transform words and sentences into dense numerical vectors that capture semantic meaning. When two texts discuss similar concepts, their embedding vectors will be close together in the vector space, even if they use completely different words. Popular embedding models include BGE (BAAI General Embedding), Sentence-BERT, and E5. Vector databases like FAISS (Facebook AI Similarity Search), Pinecone, Chroma, and Weaviate power modern search and RAG applications. FAISS is an open-source library that enables efficient similarity search in high-dimensional spaces.

Section 8: Transfer Learning and Fine-Tuning
Transfer learning is a technique where a model trained on one task is repurposed for a different but related task. Instead of training from scratch, you start with a pre-trained model and adapt it to your specific use case. Fine-tuning involves taking a pre-trained model and continuing training on a smaller, task-specific dataset. This approach dramatically reduces the data and compute needed compared to training from scratch. Popular approaches include full fine-tuning where all model parameters are updated, LoRA (Low-Rank Adaptation) which adds small trainable matrices, and prompt tuning which optimizes only the input prompts.

Section 9: AI Ethics and Responsible AI
AI ethics involves addressing the moral and social implications of AI systems. Key concerns include bias in training data leading to discriminatory outcomes against certain demographic groups, lack of transparency in AI decision-making often called the black box problem, privacy concerns from extensive data collection and surveillance, potential job displacement as AI automates more tasks, and environmental impact from the massive computational resources needed for training large models. Responsible AI development requires diverse development teams, thorough bias testing, interpretability tools like LIME and SHAP, and regulatory compliance such as the EU AI Act.

Section 10: AI in Healthcare
AI is transforming healthcare through numerous applications. Machine learning models can detect diseases from medical images like X-rays, CT scans, and MRIs with accuracy comparable to experienced radiologists. Natural language processing helps extract insights from electronic health records, clinical notes, and medical literature. Drug discovery has been accelerated by AI models that can predict molecular properties and identify potential drug candidates. Personalized medicine uses patient data to tailor treatments to individual genetic profiles. AI-powered chatbots provide initial triage and health information to patients, reducing the burden on healthcare systems.
"""

In [5]:
document = Document(
    page_content=sample_text,
    metadata={"source": "AI_Textbook", "chapter": "Overview", "author": "Teaching Guide"}
)


In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=20

 
)

In [7]:
chunks = text_splitter.split_documents([document])

print("=" * 60)
print(f"CHUNKING RESULTS")
print("=" * 60)
print(f"Original document length : {len(document.page_content)} characters")
print(f"Chunk size               : 500 characters")
print(f"Chunk overlap            : 50 characters")
print(f"Number of chunks created : {len(chunks)}")
print("=" * 60)

for i, chunk in enumerate(chunks):
    print(f"\n{'━' * 60}")
    print(f"  CHUNK {i+1}/{len(chunks)}")
    print(f"  Length: {len(chunk.page_content)} chars | Start Index: {chunk.metadata.get('start_index', 'N/A')}")
    print(f"{'━' * 60}")
    print(chunk.page_content)

print(f"\n{'=' * 60}")
print("All chunks displayed above. Each chunk is a self-contained piece of text.")
print(f"{'=' * 60}")

CHUNKING RESULTS
Original document length : 7111 characters
Chunk size               : 500 characters
Chunk overlap            : 50 characters
Number of chunks created : 31

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  CHUNK 1/31
  Length: 49 chars | Start Index: N/A
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Artificial Intelligence: A Comprehensive Overview

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  CHUNK 2/31
  Length: 43 chars | Start Index: N/A
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Section 1: What is Artificial Intelligence?

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  CHUNK 3/31
  Length: 496 chars | Start Index: N/A
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Artificial Intelligence (AI) is the simulation of human intelligence processes by computer systems. These processes include learning (acquiring information and rules for using it), reasoning (using rules to rea

In [8]:
print("CHUNK OVERLAP DEMONSTRATION")
print("=" * 60)

if len(chunks) >= 2:
    chunk1_text = chunks[0].page_content
    chunk2_text = chunks[1].page_content

    print(f"\nChunk 1 (last 100 chars):")
    print(f"  ...{chunk1_text[-100:]}")

    print(f"\nChunk 2 (first 100 chars):")
    print(f"  {chunk2_text[:100]}...")

    overlap = ""
    for i in range(min(len(chunk1_text), len(chunk2_text)), 0, -1):
        if chunk1_text.endswith(chunk2_text[:i]):
            overlap = chunk2_text[:i]
            break

    if overlap:
        print(f"\nOverlapping text ({len(overlap)} chars):")
        print(f"  >>> '{overlap}'")
    else:
        print("\nNo exact overlap found (splitter may have adjusted boundaries)")

print("\nWhy overlap matters:")
print("  - Prevents cutting sentences in half")
print("  - Maintains context across chunk boundaries")
print("  - Improves retrieval accuracy")

CHUNK OVERLAP DEMONSTRATION

Chunk 1 (last 100 chars):
  ...Artificial Intelligence: A Comprehensive Overview

Chunk 2 (first 100 chars):
  Section 1: What is Artificial Intelligence?...

No exact overlap found (splitter may have adjusted boundaries)

Why overlap matters:
  - Prevents cutting sentences in half
  - Maintains context across chunk boundaries
  - Improves retrieval accuracy


In [9]:
model_name = "BAAI/bge-small-en-v1.5"

embedding_model = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
sample_chunk = chunks[0]


chunk_embedding = embedding_model.embed_query(sample_chunk.page_content)



STEP 1: The Raw Text Chunk
Artificial Intelligence: A Comprehensive Overview

Length: 49 characters
Words : 5 words


STEP 2: The Embedding Vector
Type       : <class 'list'>
Dimensions : 384

First 20 values of the embedding vector:
  [-0.013946113176643848, -0.0005354290478862822, 0.02977885864675045, -0.0474194698035717, 0.004391423426568508, 0.03951612859964371, 0.017003947868943214, 0.04708223044872284, 0.054696764796972275, -0.006212136708199978, 0.024196287617087364, -0.025834672152996063, 0.032457541674375534, 0.030115660279989243, 0.057646725326776505, 0.016128038987517357, 0.013946414925158024, 0.036557842046022415, 0.017473367974162102, -0.04098493233323097]

Last 5 values:
  [0.02973991073668003, -0.04196280986070633, 0.039230335503816605, 0.017045946791768074, -0.04142638295888901]

Min value  : -0.287608
Max value  : 0.273584
Mean value : 0.000373
Norm (L2)  : 1.000000 (should be ~1.0 since normalized)

SUMMARY: Text Chunk → 384-dimensional numerical vector!
This vector c

In [42]:
pwd

'C:\\Users\\SSD'

In [13]:
import time

print("Creating vector store from chunks...")
print(f"Number of chunks to embed: {len(chunks)}")
print()

start_time = time.time()
vectorstore = FAISS.from_documents(chunks, embedding_model)
elapsed = time.time() - start_time

print("=" * 60)
print("VECTOR STORE CREATED SUCCESSFULLY!")
print("=" * 60)
print(f"Number of vectors stored  : {vectorstore.index.ntotal}")
print(f"Vector dimension          : {vectorstore.index.d}")
print(f"Index type                : {type(vectorstore.index).__name__}")
print(f"Time to create            : {elapsed:.2f} seconds")
print(f"Storage location          : IN-MEMORY (RAM)")
print(f"Vectorstore type          : {type(vectorstore).__name__}")
print("=" * 60)

print("\nWhere is the data stored?")
print("-" * 60)
print(f"  vectorstore object ID   : {id(vectorstore)}")
print(f"  FAISS index object      : {vectorstore.index}")
print(f"  Document store          : {type(vectorstore.docstore).__name__}")
print(f"  Number of docs in store : {len(vectorstore.docstore._dict)}")
print("\n  The embeddings live in RAM as a numpy array inside the FAISS index.")
print("  The original text is stored in the docstore (a Python dictionary).")

Creating vector store from chunks...
Number of chunks to embed: 31

VECTOR STORE CREATED SUCCESSFULLY!
Number of vectors stored  : 31
Vector dimension          : 384
Index type                : IndexFlatL2
Time to create            : 0.55 seconds
Storage location          : IN-MEMORY (RAM)
Vectorstore type          : FAISS

Where is the data stored?
------------------------------------------------------------
  vectorstore object ID   : 2037691325408
  FAISS index object      : <faiss.swigfaiss_avx2.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x000001DA6D7DE2B0> >
  Document store          : InMemoryDocstore
  Number of docs in store : 31

  The embeddings live in RAM as a numpy array inside the FAISS index.
  The original text is stored in the docstore (a Python dictionary).


In [15]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

In [16]:
query = "What is deep learning and how does it work?"

In [17]:
retrieved_docs = retriever.invoke(query)

In [19]:
print(f"\nNumber of chunks retrieved: {len(retrieved_docs)}")

for i, doc in enumerate(retrieved_docs):
    print(f"\n{'━' * 60}")
    print(f"  RETRIEVED CHUNK {i+1}")
    print(f"  Source: {doc.metadata.get('source', 'N/A')}")
    print(f"  Start Index: {doc.metadata.get('start_index', 'N/A')}")
    print(f"{'━' * 60}")
    print(doc.page_content)

print(f"\n{'=' * 60}")
print("Notice how the retriever found chunks about deep learning")
print("and neural networks — the most relevant to our query!")
print(f"{'=' * 60}")


Number of chunks retrieved: 4

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  RETRIEVED CHUNK 1
  Source: AI_Textbook
  Start Index: N/A
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Deep Learning is a subset of machine learning that uses artificial neural networks with multiple layers to model complex patterns. A neural network consists of layers of interconnected nodes (neurons) that process information. Key architectures include Convolutional Neural Networks (CNNs) which are designed for processing grid-like data such as images, Recurrent Neural Networks (RNNs) which handle sequential data like text and time series, and Transformers which use self-attention mechanisms

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  RETRIEVED CHUNK 2
  Source: AI_Textbook
  Start Index: N/A
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Section 3: Deep Learning and Neural Networks

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [20]:
query = "What is deep learning and how does it work?"

results_with_scores = vectorstore.similarity_search_with_score(query, k=6)

print("RETRIEVAL WITH SIMILARITY SCORES")
print("=" * 60)
print(f"Query: \"{query}\"")
print(f"Showing top 6 results with their distance scores")
print("(Lower distance = Higher relevance)")
print("=" * 60)

for i, (doc, score) in enumerate(results_with_scores):
    relevance = "HIGH" if score < 0.8 else ("MEDIUM" if score < 1.2 else "LOW")
    print(f"\n--- Rank {i+1} | L2 Distance: {score:.4f} | Relevance: {relevance} ---")
    print(f"Text: {doc.page_content[:200]}...")
    print(f"Metadata: {doc.metadata}")

print(f"\n{'=' * 60}")
print("Observe how scores increase (less relevant) as we go down the list.")
print("The retriever returns chunks ordered by relevance!")
print(f"{'=' * 60}")

RETRIEVAL WITH SIMILARITY SCORES
Query: "What is deep learning and how does it work?"
Showing top 6 results with their distance scores
(Lower distance = Higher relevance)

--- Rank 1 | L2 Distance: 0.3617 | Relevance: HIGH ---
Text: Deep Learning is a subset of machine learning that uses artificial neural networks with multiple layers to model complex patterns. A neural network consists of layers of interconnected nodes (neurons)...
Metadata: {'source': 'AI_Textbook', 'chapter': 'Overview', 'author': 'Teaching Guide'}

--- Rank 2 | L2 Distance: 0.5344 | Relevance: HIGH ---
Text: Section 3: Deep Learning and Neural Networks...
Metadata: {'source': 'AI_Textbook', 'chapter': 'Overview', 'author': 'Teaching Guide'}

--- Rank 3 | L2 Distance: 0.5732 | Relevance: HIGH ---
Text: Computer Vision enables machines to interpret and make decisions based on visual data. Major applications include facial recognition for security and authentication, autonomous vehicles for detecting ...
Metadata: {'s

In [21]:
queries = [
    "What are the ethical concerns in AI?",
    "How is AI used in healthcare?",
    "What is transfer learning and fine-tuning?",
    "Explain RAG and vector databases",
]

print("MULTI-QUERY RETRIEVAL TEST")
print("=" * 60)

for query in queries:
    print(f"\n{'━' * 60}")
    print(f"QUERY: \"{query}\"")
    print(f"{'━' * 60}")

    results = vectorstore.similarity_search_with_score(query, k=2)

    for i, (doc, score) in enumerate(results):
        print(f"  [{i+1}] Score: {score:.4f}")
        print(f"      Chunk: \"{doc.page_content[:120]}...\"")
        print()

print("=" * 60)
print("Each query retrieves DIFFERENT chunks — exactly the relevant ones!")
print("=" * 60)

MULTI-QUERY RETRIEVAL TEST

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
QUERY: "What are the ethical concerns in AI?"
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  [1] Score: 0.2154
      Chunk: "AI ethics involves addressing the moral and social implications of AI systems. Key concerns include bias in training dat..."

  [2] Score: 0.3149
      Chunk: "Section 9: AI Ethics and Responsible AI..."


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
QUERY: "How is AI used in healthcare?"
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  [1] Score: 0.3177
      Chunk: "Section 10: AI in Healthcare..."

  [2] Score: 0.3770
      Chunk: "AI is transforming healthcare through numerous applications. Machine learning models can detect diseases from medical im..."


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
QUERY: "What is transfer learning and fine-tuning?"
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
